#🥇 Camada Gold

A camada **Gold** representa o nível final da Arquitetura Medalhão.
É onde os dados, já limpos e estruturados na camada Silver, são transformados em informação de negócio: agregações, métricas, indicadores e visões analíticas.

Aqui nasce o modelo analítico que alimenta dashboards, relatórios e aplicações de dados.

##1º Projeto — Área de Logística (Vendas por Localidade)
A área de Logística deseja identificar quais cidades e estados concentram
mais vendas, para otimizar rotas de entrega e centros de distribuição.

###1.1 Criação da tabela _gold.ft_vendas_consumidor_local_

Cada linha representa um pedido realizado por um consumidor
Não queremos criar o consolidado nessa fato, uma vez que irão surgir
novos pedidos, queremos manter o histórico deles, logo a informação
consolidada deve constar apenas na view


As informações devem ser obtidas a partir de:

`silver.ft_pedido_total` (valor total do pedido)

`silver.ft_consumidores` (cidade e estado do consumidor)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window 
from pyspark.sql.types import DecimalType

catalogo = "medalhao"
schema = "gold"

pedido_total = spark.table("medalhao.silver.ft_pedido_total")   
consumidores = spark.table("medalhao.silver.ft_consumidores")   

vendas_enriquecidas = (
    pedido_total.alias("p")
    .join(consumidores.alias("c"), on="id_consumidor", how="left") # left pra pegar todos os pedidos independentemente
    .select(
        F.col("p.id_pedido"),
        F.col("p.id_consumidor"),
        F.col("p.valor_total_pago_brl"),
        F.col("c.cidade"),
        F.col("c.estado"),
        F.to_date(F.col("p.data_pedido")).alias("data_pedido")
    )
)


vendas_enriquecidas.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema}.ft_vendas_consumidor_local")


###1.2 Criação da view _gold.view_total_compras_por_consumidor_

Consolida o total de compras e o valor total vendido por **cidade e estado**,
permitindo identificar localidades com maior concentração de vendas.

In [0]:
# aqui vamos precisar agrupar cidade e estado e usar funções de agregação para quantidade de vendas e para os valores
spark.sql("""
    CREATE OR REPLACE VIEW medalhao.gold.vw_vendas_consumidor_local AS
    SELECT
        estado,
        cidade,
        COUNT(DISTINCT id_pedido) AS quantidade_vendas,   -- poderia ser COUNT(*) ja que não temos pedidos duplicados 
        SUM(valor_total_pago_brl) AS valor_total_localidade
    FROM medalhao.gold.ft_vendas_consumidor_local
    GROUP BY estado, cidade
    ORDER BY quantidade_vendas DESC
""")

Crie uma consulta SQL para exibir o total de vendas por estado


In [0]:
%sql
-- consulta simples 
SELECT estado, SUM(quantidade_vendas) AS quantidade_vendas
FROM medalhao.gold.vw_vendas_consumidor_local
GROUP BY estado
ORDER BY quantidade_vendas DESC


##2º Projeto — Área de Logística (Análise de Atrasos de Entregas)
A equipe de Logística está enfrentando aumento nos índices de atraso e quer
identificar as regiões e vendedores mais associados aos atrasos.
O objetivo agora é identificar os estados e cidades com mais atrasos e
entender quais vendedores estão mais associados a essas ocorrências,
permitindo detectar possíveis gargalos na cadeia de entrega.

### 2.1 Criação da tabela _gold.ft_atrasos_pedidos_local_vendedor_
Cada linha representa um pedido com suas informações logísticas básicas.

As informações devem ser obtidas a partir de:

`silver.ft_pedidos`

`silver.ft_consumidores`

`silver.ft_itens_pedidos`

In [0]:
df_pedidos = spark.table("medalhao.silver.ft_pedidos")
df_consumidores = spark.table("medalhao.silver.ft_consumidores")
df_itens_pedidos = spark.table("medalhao.silver.ft_itens_pedidos")

# filtrando algumas tabelas com as informações que queremos

df_pedidos_logistica = df_pedidos.select(
    "id_pedido",
    "id_consumidor",
    F.col("entrega_no_prazo"),
    F.col("tempo_entrega_dias"),
    F.col("tempo_entrega_estimado_dias")
)

# renomeando as colunas de localização para evitar conflito após o JOIN
df_localizacao = df_consumidores.select(
    "id_consumidor",
    F.col("cidade").alias("cidade_consumidor"),
    F.col("estado").alias("estado_consumidor")
)

# primeiro join de pedidos com os itens dos pedidos(inner join para manter apenas os pedidos que tem itens)
df_join1 = df_pedidos_logistica.join(
    df_itens_pedidos,
    on="id_pedido",
    how="inner"
)

# segundo join de pedidos e itens com a localização dos consumidores
df_gold_temp = df_join1.join(
    df_localizacao,
    on="id_consumidor",
    how="left"
)

# select final com as colunas pedidas
df_ft_atrasos_pedidos_local_vendedor = df_gold_temp.select(
    F.col("id_pedido"),
    F.col("id_vendedor"),
    F.col("id_consumidor"),
    
    # Colunas de Logística
    F.col("entrega_no_prazo"),
    F.col("tempo_entrega_dias"),
    F.col("tempo_entrega_estimado_dias"),
    
    # Colunas de Localização
    F.col("cidade_consumidor").alias("cidade"),
    F.col("estado_consumidor").alias("estado")
).orderBy(F.col("tempo_entrega_dias").desc())

df_ft_atrasos_pedidos_local_vendedor.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema}.ft_atrasos_pedidos_local_vendedor")

df_ft_atrasos_pedidos_local_vendedor.display()

###2.2 Criação das Views Analítica 
####2.2.1 gold.view_tempo_medio_entrega_localidade


In [0]:
spark.sql("""
CREATE OR REPLACE VIEW medalhao.gold.view_tempo_medio_entrega_localidade AS
SELECT
    cidade,
    estado,
    -- calcula a média do tempo real de entrega 
    CAST(ROUND(AVG(tempo_entrega_dias), 2) AS DECIMAL(10, 2)) AS tempo_medio_entrega,
    -- calcula a média do tempo estimado de entrega 
    CAST(ROUND(AVG(tempo_entrega_estimado_dias), 2) AS DECIMAL(10, 2)) AS tempo_medio_estimado,
    -- verifica se o tempo médio real é maior que o tempo médio estimado
    CASE
        WHEN AVG(tempo_entrega_dias) > AVG(tempo_entrega_estimado_dias) THEN 'SIM'
        ELSE 'NÃO'
    END AS entrega_maior_que_estimado
FROM
    medalhao.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY
    cidade,
    estado
ORDER BY
    tempo_medio_entrega DESC
""")

spark.table("medalhao.gold.view_tempo_medio_entrega_localidade").display()

####2.2.2 gold.view_vendedor_pontualidade



In [0]:
spark.sql("""
CREATE OR REPLACE VIEW medalhao.gold.view_vendedor_pontualidade AS
SELECT
    id_vendedor,
    -- conta o total de pedidos por vendedor
    COUNT(id_pedido) AS total_pedidos,
    -- conta os pedidos onde 'entrega_no_prazo' é 'Não' ou 'Não Entregue'
    SUM(CASE WHEN entrega_no_prazo IN ('Não', 'Não Entregue') THEN 1 ELSE 0 END) AS total_atrasados,
    -- calcula o percentual de atraso
    CAST(
        (SUM(CASE WHEN entrega_no_prazo IN ('Não', 'Não Entregue') THEN 1 ELSE 0 END) * 100.0)
        / COUNT(id_pedido)
    AS DECIMAL(5, 2)) AS percentual_atraso
FROM
    medalhao.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY
    id_vendedor
ORDER BY
    total_atrasados DESC
""")

spark.table("medalhao.gold.view_vendedor_pontualidade").display()

##3º Projeto — Área Comercial (Análises de Vendas por Período)
A área comercial necessita acompanhar a evolução das vendas e criar
análises temporais e de desempenho.


###3.1 Criação da Dimensão de Tempo — gold.dm_tempo
Será necessário criar uma dimensão, para que auxilie nas análises temporais
em diferentes granularidades (ano, trimestre, mês, semana, dia, etc.). Utilize as
funções explode e sequence para gerar os valores entre as datas de início e fim.


In [0]:
df_pedidos = spark.table("medalhao.silver.ft_pedidos")

# setando intervalo de datas (data de compra mais antiga e mais recente)
min_max_dates = df_pedidos.select(
    F.min(F.to_date(F.col("pedido_compra_timestamp"))).alias("start_date"),
    F.max(F.to_date(F.col("pedido_compra_timestamp"))).alias("end_date")
).collect()[0]

start_date = min_max_dates["start_date"]
end_date = min_max_dates["end_date"]

# cria tabela com todas as datas no range que setamos
df_datas_base = spark.range(1).select(
    F.explode(
        F.expr(f"sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)")
    ).alias("sk_tempo")
)

# select com as colunas requeridas 
df_dm_tempo = df_datas_base.withColumn("data", F.col("sk_tempo")).select(
    F.col("sk_tempo"),
    # uso de funções de calendário para extrair as novas colunas
    F.year("data").alias("ano"),
    F.quarter("data").alias("trimestre"),
    F.month("data").alias("mes"),
    F.dayofmonth("data").alias("dia"),
    F.weekofyear("data").alias("semana_do_ano"),

    F.dayofweek("data").alias("dia_da_semana_num"),
    
    F.when(F.dayofweek("data") == 1, "Domingo")
     .when(F.dayofweek("data") == 2, "Segunda-feira")
     .when(F.dayofweek("data") == 3, "Terça-feira")
     .when(F.dayofweek("data") == 4, "Quarta-feira")
     .when(F.dayofweek("data") == 5, "Quinta-feira")
     .when(F.dayofweek("data") == 6, "Sexta-feira")
     .when(F.dayofweek("data") == 7, "Sábado")
     .alias("dia_da_semana_nome"),
     
    F.when(F.dayofweek("data").isin([1, 7]), "Sim").otherwise("Não").alias("eh_fim_de_semana"),
    
    # Nome do Mês (em Português)
    F.when(F.month("data") == 1, "Janeiro")
     .when(F.month("data") == 2, "Fevereiro")
     .when(F.month("data") == 3, "Março")
     .when(F.month("data") == 4, "Abril")
     .when(F.month("data") == 5, "Maio")
     .when(F.month("data") == 6, "Junho")
     .when(F.month("data") == 7, "Julho")
     .when(F.month("data") == 8, "Agosto")
     .when(F.month("data") == 9, "Setembro")
     .when(F.month("data") == 10, "Outubro")
     .when(F.month("data") == 11, "Novembro")
     .when(F.month("data") == 12, "Dezembro")
     .alias("mes_nome")
)

df_dm_tempo.write.mode("overwrite").format("delta").saveAsTable(f"{catalogo}.{schema}.dm_tempo")

df_dm_tempo.orderBy("sk_tempo").display()

###3.2 Criação da Fato gold.ft_vendas_geral
Integra informações de várias áreas da empresa.


In [0]:
df_pedidos = spark.table("medalhao.silver.ft_pedidos")
df_itens = spark.table("medalhao.silver.ft_itens_pedidos")
df_consumidores = spark.table("medalhao.silver.ft_consumidores") 
df_cotacao = spark.table("medalhao.silver.dm_cotacao_dolar")
df_avaliacoes = spark.table("medalhao.silver.ft_avaliacoes_pedidos")
 
# preparar df_pedidos com as colunas que queremos
df_pedidos_prep = df_pedidos.select(
    F.col("id_pedido"),
    F.col("id_consumidor").alias("fk_cliente"),
    F.col("status").alias("status_pedido"),
    F.to_date(F.col("pedido_compra_timestamp")).alias("fk_tempo"),
    F.col("tempo_entrega_dias"),
    F.col("entrega_no_prazo")
)

# preparar df_itens - USAR try_cast para os valores decimais
df_itens_prep = df_itens.select(
    F.col("id_pedido"),
    F.col("id_item").cast("string"),
    F.col("id_produto").alias("fk_produto"),
    F.col("id_vendedor").alias("fk_vendedor"),
    F.expr("try_cast(preco_BRL as decimal(12,2))").alias("valor_produto_brl"),
    F.expr("try_cast(preco_frete as decimal(12,2))").alias("valor_frete_brl")
).withColumn(
    "valor_total_item_brl", 
    F.col("valor_produto_brl") + F.col("valor_frete_brl")
)

# preparar df_cotacao
df_cotacao_prep = df_cotacao.select(
    # renomeamos a data aqui para evitar conflito com a coluna 'data' do JOIN
    F.col("cotacao_dolar").cast(DecimalType(8,4)).alias("cotacao_usada"), 
    F.col("data").alias("data_cotacao")
)


# preparar df_avaliacoes (calcular média das avaliações por pedido)
df_avaliacoes_prep = df_avaliacoes.groupBy("id_pedido").agg(
    F.avg("avaliacao").alias("avaliacao_pedido")
)

# juntando todas as tabelas
df_vendas_geral_temp = (
    df_itens_prep
    .join(df_pedidos_prep, "id_pedido", "inner")
    .join(df_avaliacoes_prep, "id_pedido", "left")
    .join(
        df_cotacao_prep,
        df_pedidos_prep["fk_tempo"] == df_cotacao_prep["data_cotacao"],
        "left" # deixa todos os pedidos, mesmo sem cotação
    )
    # Remove a coluna de data duplicada usada no JOIN
    .drop(df_cotacao_prep["data_cotacao"]) 
)

# 6. Cálculo das Métricas USD usando a coluna cotacao_usada
df_vendas_geral = df_vendas_geral_temp.select(
    "id_pedido",
    "id_item",
    "fk_cliente",
    "fk_produto",
    "fk_vendedor",
    "fk_tempo",
    "status_pedido",
    "tempo_entrega_dias",
    "entrega_no_prazo",
    "valor_produto_brl",
    "valor_frete_brl",
    "valor_total_item_brl",

    # cotação usada no cálculo
    F.round(F.col("cotacao_usada"), 2).alias("cotacao_dolar"),
    
    # valores USD calculados usando a coluna cotacao_usada
    F.round(F.col("valor_produto_brl") / F.col("cotacao_usada"), 2).alias("valor_produto_usd"),
    F.round(F.col("valor_frete_brl") / F.col("cotacao_usada"), 2).alias("valor_frete_usd"),
    F.round(F.col("valor_total_item_brl") / F.col("cotacao_usada"), 2).alias("valor_total_item_usd"),

    F.col("avaliacao_pedido")
)

df_vendas_geral.write.mode("overwrite").format("delta").saveAsTable("medalhao.gold.ft_vendas_geral")

df_vendas_geral.display()

###3.3 Criação da view gold.view_vendas_por_periodo
A área de Business Intelligence (BI) e Planejamento Comercial do Ecommerce precisa de uma visão temporal consolidada que permita avaliar o
comportamento das vendas ao longo do tempo: por ano, trimestre, mês e dia
da semana.
Essa análise servirá como base para dashboards de desempenho e para prever
padrões sazonais (como aumento nas vendas em meses específicos ou dias
com mais pedidos).


In [0]:
spark.sql("""
CREATE OR REPLACE VIEW medalhao.gold.view_vendas_por_periodo AS
SELECT
    T2.ano,
    T2.trimestre,
    T2.mes,
    T2.mes_nome,
    T2.dia,
    T2.dia_da_semana_num,
    
    COUNT(DISTINCT T1.id_pedido) AS total_pedidos, -- conta todos pedidos
    COUNT(*) AS total_itens, -- conta todos itens
    
    CAST(SUM(T1.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
    CAST(SUM(T1.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
    CAST(AVG(T1.valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
    CAST(AVG(T1.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
    
FROM
    medalhao.gold.ft_vendas_geral T1
INNER JOIN
    medalhao.gold.dm_tempo T2 ON T1.fk_tempo = T2.sk_tempo
GROUP BY
    T2.ano, T2.trimestre, T2.mes, T2.mes_nome, T2.dia, T2.dia_da_semana_num
ORDER BY
    T2.ano, T2.mes, T2.dia
""")
# alguns valores da receita em dolar vao estar null por causa do intervalo que setei no landing to bronze para cotacao_dolar

spark.table("medalhao.gold.view_vendas_por_periodo").display()

####3.3.1 Consultas Analíticas
Com a criação da view gold.view_vendas_por_periodo , o time de Análise Comercial
deseja entender em quais dias da semana e meses a empresa mais fatura e
onde estão as melhores oportunidades de crescimento. Crie querrys na view,
que respondam as seguintes perguntas


1. Qual é o dia da semana com maior receita total em reais (receita_total_brl)?

In [0]:
%sql
SELECT 
    T2.dia_da_semana_nome,
    CAST(SUM(T1.receita_total_brl) AS DECIMAL(12, 2)) AS receita_total_acumulada
FROM 
    medalhao.gold.view_vendas_por_periodo T1
INNER JOIN
    medalhao.gold.dm_tempo T2 ON T1.dia_da_semana_num = T2.dia_da_semana_num
GROUP BY 
    T2.dia_da_semana_num, T2.dia_da_semana_nome
ORDER BY 
    receita_total_acumulada DESC
LIMIT(1)

####3.3.1 Consultas Analíticas
2. Considerando o último ano disponível na dimensão de tempo, qual foi o mês com maior ticket médio (ticket_medio_brl)?

In [0]:
%sql
WITH UltimoAno AS ( 
    SELECT MAX(ano) AS max_ano FROM medalhao.gold.view_vendas_por_periodo -- CTE para obter o ano mais recente
)
SELECT 
    T1.ano,
    T1.mes_nome,
    T1.mes, 
    ROUND(AVG(T1.ticket_medio_brl), 2) AS ticket_medio_brl
FROM 
    medalhao.gold.view_vendas_por_periodo T1, 
    UltimoAno UA
WHERE 
    T1.ano = UA.max_ano
GROUP BY T1.ano, T1.mes, T1.mes_nome
ORDER BY 
    ticket_medio_brl DESC
LIMIT(1)

##3.4. Criação da gold.view_top_produto
A área de Gestão de Produtos e Comercial do E-commerce deseja
compreender quais produtos e categorias geram mais receita e possuem
melhor desempenho de vendas e avaliação.
Essas informações são fundamentais para definir estratégias de precificação,
estoque e priorização de campanhas de marketing.


In [0]:
spark.sql("""
CREATE OR REPLACE VIEW medalhao.gold.view_top_produto AS
SELECT
    T1.fk_produto AS id_produto,
    T2.categoria_produto, 

    COUNT(*) AS quantidade_vendida,
    COUNT(DISTINCT T1.id_pedido) AS total_pedidos,
    
    CAST(SUM(T1.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_brl,
    CAST(SUM(T1.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_usd,
    CAST(AVG(T1.valor_produto_brl) AS DECIMAL(12,2)) AS preco_medio_brl,
    CAST(AVG(T1.avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media,
    CAST(AVG(T2.peso_produto_gramas) AS DECIMAL(8,2)) AS peso_medio_gramas
    
FROM
    medalhao.gold.ft_vendas_geral T1
INNER JOIN
    medalhao.silver.ft_produtos T2 ON T1.fk_produto = T2.id_produto
GROUP BY
    T1.fk_produto,
    T2.categoria_produto
ORDER BY
    receita_brl DESC
""")

spark.table("medalhao.gold.view_top_produto").display()

##3.5 Criação da view_vendas_produtos_esteticos
A área de Fashion do E-commerce deseja acompanhar o desempenho de
vendas dessa categoria específica ao longo do tempo.

 O objetivo é entender a
evolução da receita, o volume de pedido e as avaliações médias dos clientes
mês a mês — permitindo ajustar estratégias de marketing e estoque conforme
a demanda.
⚠️ Para a criação dessa view, será obrigatório o uso de CTE

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW medalhao.gold.view_vendas_produtos_esteticos AS

-- criando a CTE das vendas com produtos fashion
WITH VendasFashion AS (
    SELECT
        T1.id_pedido,
        T1.valor_total_item_brl,
        T1.valor_total_item_usd,
        T1.avaliacao_pedido,
        T2.categoria_produto,
        T3.ano,
        T3.mes
    FROM
        medalhao.gold.ft_vendas_geral T1
    INNER JOIN
        medalhao.silver.ft_produtos T2 ON T1.fk_produto = T2.id_produto
    INNER JOIN
        medalhao.gold.dm_tempo T3 ON T1.fk_tempo = T3.sk_tempo
    WHERE
        T2.categoria_produto LIKE 'fashion%' -- filtrando apenas produtos que possuam fashion no começo do nome
)

-- agregação mensal por categoria
SELECT
    ano,
    mes,
    categoria_produto,
    
    COUNT(DISTINCT id_pedido) AS total_pedidos,
    COUNT(*) AS total_itens_vendidos,

    CAST(SUM(valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
    CAST(SUM(valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
    CAST(AVG(valor_total_item_brl) AS DECIMAL(12,2)) AS ticket_medio_brl,
    CAST(AVG(valor_total_item_usd) AS DECIMAL(12,2)) AS ticket_medio_usd,
    CAST(AVG(avaliacao_pedido) AS DECIMAL(3,2)) AS avaliacao_media
    
FROM
    VendasFashion
GROUP BY
    ano, mes, categoria_produto
ORDER BY
    ano, mes, categoria_produto
""")

spark.table("medalhao.gold.view_vendas_produtos_esteticos").display()